# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [1]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


commit : d27c1c0
GPU    : Tesla T4
VRAM   : 14.6 GB


In [2]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 3.96s

7 passed · 0 warnings · 0 failures

Ready to submit.


In [7]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


Tesla T4 (cuda, sm_75, 14.6 GB) -> precision=fp16
  note: this GPU predates Ampere, so it has NO bfloat16. Using fp16 with gradient scaling instead. Tutorials that hardcode bf16=True fail here. 

tier=T4  mask=assistant-only  eval_limit=full

NB1 — data, chat template & loss mask
tier=T4  model=unsloth/Qwen3.5-4B  max_length=1024
250 mẫu huấn luyện
{
  "instruction": "Phân loại ticket chăm sóc khách hàng sau thành JSON với đúng 4 khóa: intent, urgency, product, sentiment. Chỉ trả về JSON, không giải thích.\n\nintent thuộc: doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin\nurgency thuộc: cao | trung_binh | thap\nsentiment thuộc: tieu_cuc | trung_tinh | tich_cuc\nproduct: tên sản phẩm xuất hiện trong ticket.",
  "input": "Alo sh
eos_token: <|im_end|>
VERDICT: reasoning preserved — safe to train on traces

--- chuỗi đã render ---
<|im_start|>user
2+2?<|im_end|>
<|im_start|>assistant
<think>
buoc 1: kiem tra. buoc 2: tra loi.
</think>

4<|im_end|>

mode = assistant-only   su

In [8]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 3.36s
[  ok  ] results/template_check.json                      
[  ok  ] results/mask_proof.json                          
[  ok  ] results/token_stats.json                         
[  ok  ] results/baselines_frozen.json                    
[  ok  ] results/runs.csv                                 
[  ok  ] results/verdict.json                             
[  ok  ] results/autopsy.json                             
[  ok  ] submission/REPORT.md                             
[ FAIL ] REPORT.md filled in                     

In [10]:
#!/usr/bin/env python3
"""
Run this IN COLAB (not locally) to package and download all artifacts.
Copy-paste into a Colab cell and run.
"""
import os
import zipfile
import subprocess
from pathlib import Path

# Find the repo root in Colab
possible_paths = [
    "/content/Day21-Track3-Finetuning-Lab",
    "/content/drive/MyDrive/Day21-Track3-Finetuning-Lab",
    "/home/user/Day21-Track3-Finetuning-Lab",
]
repo_root = None
for p in possible_paths:
    if Path(p).exists():
        repo_root = Path(p)
        break

if repo_root is None:
    # Try to find it via git
    result = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
    if result.returncode == 0:
        repo_root = Path(result.stdout.strip())

if repo_root is None:
    raise RuntimeError("Could not find repo root. Run this in Colab after cloning the repo.")

print(f"Repo root: {repo_root}")

# Files to package
files_to_pack = [
    # Results
    "results/baselines_frozen.json",
    "results/runs.csv",
    "results/verdict.json",
    "results/autopsy.json",
    "results/qualitative.json",
    # Adapters
    "adapters/correct/adapter_model.safetensors",
    "adapters/correct/adapter_config.json",
    "adapters/attn_only/adapter_model.safetensors",
    "adapters/attn_only/adapter_config.json",
    "adapters/wrong_lr/adapter_model.safetensors",
    "adapters/wrong_lr/adapter_config.json",
    "adapters/qlora/adapter_model.safetensors",
    "adapters/qlora/adapter_config.json",
]

existing_files = []
missing_files = []
for rel in files_to_pack:
    p = repo_root / rel
    if p.exists():
        existing_files.append(p)
    else:
        missing_files.append(rel)

if missing_files:
    print(f"⚠️ Missing (not generated yet): {missing_files}")

if not existing_files:
    raise RuntimeError("No artifact files found. Run NB2-NB5 first.")

# Create zip
zip_path = repo_root / "lab21_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in existing_files:
        arcname = p.relative_to(repo_root)
        zf.write(p, arcname)
        print(f"  Added: {arcname}")

print(f"\n✅ Created: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(str(zip_path))
    print("📥 Download started in browser...")
except ImportError:
    print("Not in Colab environment. Zip saved at:", zip_path)

Repo root: /content/Day21-Track3-Finetuning-Lab
  Added: results/baselines_frozen.json
  Added: results/runs.csv
  Added: results/verdict.json
  Added: results/autopsy.json
  Added: results/qualitative.json
  Added: adapters/correct/adapter_model.safetensors
  Added: adapters/correct/adapter_config.json
  Added: adapters/attn_only/adapter_model.safetensors
  Added: adapters/attn_only/adapter_config.json
  Added: adapters/wrong_lr/adapter_model.safetensors
  Added: adapters/wrong_lr/adapter_config.json
  Added: adapters/qlora/adapter_model.safetensors
  Added: adapters/qlora/adapter_config.json

✅ Created: /content/Day21-Track3-Finetuning-Lab/lab21_artifacts.zip (477.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started in browser...
